# 第 5 课：从零实现 JAX PPO（连续动作）

**学习目标**：独立运行一个具备随机策略、Critic、GAE、clipped surrogate、mini-batch 与 Adam 的 PPO 实例。

> 这节是整门课程的核心。我们故意选用一个简单的 1D 受控动力学，来验证算法本身。它**不是二维圆柱绕流**；后续只替换环境、观测与奖励接口。

## 5.1 PPO 从哪来？

旧策略采样 $(s_t,a_t)$，用旧策略下的优势估计 $\hat A_t$ 判断这个动作好不好。概率比值：

$$ r_t(\theta)=\frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\rm old}}(a_t|s_t)}=\exp\big(\log\pi_\theta-\log\pi_{\theta_{\rm old}}\big). $$

PPO-Clip 希望最大化：

$$L^{\rm CLIP}(\theta)=\mathbb E_t\left[\min\left(r_t(\theta)\hat A_t,\operatorname{clip}(r_t(\theta),1-\epsilon,1+\epsilon)\hat A_t\right)\right].$$

通常同时优化 $L_V=\frac12\mathbb E_t[V_\phi(s_t)-\hat V_t]^2$，可选加熵奖励避免过早丢失探索。这里只写最核心的 PPO-Clip，不把它误称为精确 trust-region 方法。

**科研提醒**：本实验用一个可解析的线性高斯 Actor 和二次特征 Critic，是为了把优化过程看清楚。真实流场策略通常需要 MLP/CNN，并可能使用传感器特征、历史观测或 latent state。

## 5.2 定义被控制的系统

$$ x_{t+1}=0.82x_t+0.25\,\operatorname{clip}(a_t,-2,2)+\xi_t,\quad \xi_t\sim\mathcal N(0,0.025^2). $$

$$r_t=-x_{t+1}^2-0.03\,\operatorname{clip}(a_t,-2,2)^2.$$

代理动力学在这里被写成 Python/NumPy；你后面可以换成 CFD 或神经网络推演。

动作由高斯随机策略采样，并且**保留未裁剪的随机动作计算 log-probability**，实际执行动作另行裁剪。若将裁剪后的动作直接塞进高斯密度，会改变训练目标的统计含义；更严格的有界连续动作可以改用 tanh-squashed Gaussian 并计算雅可比修正。

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

SEED, HORIZON = 2026, 28
rng = np.random.default_rng(SEED)

def env_step(x, raw_action, rng):
    applied_action = float(np.clip(raw_action, -2.0, 2.0))
    next_x = 0.82 * x + 0.25 * applied_action + rng.normal(0.0, 0.025)
    reward = -(next_x**2 + 0.03*applied_action**2)
    return float(next_x), float(reward)

def policy_mean(theta, x):
    return theta[0]*x + theta[1]

def value_fn(phi, x):
    return phi[0] + phi[1]*x + phi[2]*x**2

def normal_logp(theta, x, a):
    log_std = jnp.clip(theta[2], -2.0, 0.6)
    z = (a - policy_mean(theta, x)) / jnp.exp(log_std)
    return -0.5*z*z - log_std - 0.5*jnp.log(2*jnp.pi)

def normal_logp_numpy(theta, x, a):
    log_std = float(np.clip(theta[2], -2.0, 0.6))
    z = (a - float(theta[0]*x + theta[1]))/np.exp(log_std)
    return -0.5*z*z - log_std - 0.5*np.log(2*np.pi)

actor = jnp.array([0.0, 0.0, -0.5])  # mean = w*x+b, log_sigma
critic = jnp.zeros(3)  # V(s)=c0+c1*s+c2*s^2

## 5.3 从样本计算 GAE 与价值监督目标

短轨迹是因为采样配额被截断，并不是物理终止，所以此处 bootstrap 最后一个状态的价值。轨迹内按逆序累计：

$$\delta_t=r_t+\gamma V(s_{t+1})-V(s_t),\qquad A_t=\delta_t+\gamma\lambda A_{t+1}.$$

每条轨迹的最后一步之后，没有下一条存储优势，所以最后一项为 0。

In [ ]:
GAMMA, LAMBDA = 0.98, 0.95

def collect_episodes(actor, critic, rng, episodes=12):
    theta = np.asarray(actor)
    phi = np.asarray(critic)
    fields = {'s':[], 'a':[], 'old_logp':[], 'adv':[], 'target':[]}
    episodic_returns = []
    for _ in range(episodes):
        x = float(np.clip(rng.normal(0., 1.1), -2., 2.))
        states, actions, rewards, next_states, old_logps = [], [], [], [], []
        for t in range(HORIZON):
            mean = theta[0]*x + theta[1]
            raw_a = float(rng.normal(mean, np.exp(float(np.clip(theta[2], -2., .6)))))
            new_x, reward = env_step(x, raw_a, rng)
            states.append(x); actions.append(raw_a); rewards.append(reward)
            next_states.append(new_x); old_logps.append(normal_logp_numpy(theta, x, raw_a))
            x = new_x
        s = np.asarray(states); ns = np.asarray(next_states); r = np.asarray(rewards)
        v = phi[0] + phi[1]*s + phi[2]*s*s
        vn = phi[0] + phi[1]*ns + phi[2]*ns*ns
        delta = r + GAMMA*vn - v  # all sampled episode ends are time cutoffs
        adv = np.zeros_like(r)
        accum = 0.0
        for t in reversed(range(HORIZON)):
            accum = delta[t] + GAMMA*LAMBDA*accum
            adv[t] = accum
        for key, arr in [('s',s),('a',np.asarray(actions)),('old_logp',np.asarray(old_logps)),
                         ('adv',adv),('target',adv+v)]:
            fields[key].append(arr)
        episodic_returns.append(float(r.sum()))
    batch = {k: np.concatenate(v).astype(np.float32) for k,v in fields.items()}
    return batch, float(np.mean(episodic_returns))

batch, sampled_return = collect_episodes(actor, critic, rng)
print({k: v.shape for k,v in batch.items()})
print('initial sampled mean episode return:', round(sampled_return,3))

## 5.4 Loss、梯度与 Adam

下面用 JAX 直接对 Actor、Critic 参数求导。每次更新使用同一批旧策略采集的数据；**新一轮采样时才替换旧策略**。这里的 Critic 损失不做额外 clipping，属于可解释的教学简化版 PPO。

`jnp.minimum` 在优势为负数时也必须正确；直接把 `ratio` clip 之后乘优势会得到不等价的目标。

In [ ]:
EPS_CLIP, ENTROPY_COEF = 0.2, 0.003

def actor_loss(theta, states, actions, old_logps, advantage):
    new_logps = normal_logp(theta, states, actions)
    ratio = jnp.exp(new_logps - old_logps)
    surrogate1 = ratio * advantage
    surrogate2 = jnp.clip(ratio, 1-EPS_CLIP, 1+EPS_CLIP) * advantage
    entropy = .5*jnp.log(2*jnp.pi*jnp.e) + jnp.clip(theta[2], -2., .6)
    return -jnp.mean(jnp.minimum(surrogate1, surrogate2)) - ENTROPY_COEF*entropy

def critic_loss(phi, states, targets):
    return 0.5*jnp.mean((value_fn(phi, states) - targets)**2)

actor_grad = jax.jit(jax.value_and_grad(actor_loss))
critic_grad = jax.jit(jax.value_and_grad(critic_loss))

def adam_step(params, grad, m, v, t, lr):
    # 一维参数向量的 Adam，用于理解优化器的内部逻辑
    grad = grad / jnp.maximum(1., jnp.linalg.norm(grad)/1.0)
    m = .9*m + .1*grad
    v = .999*v + .001*(grad**2)
    mhat = m/(1-.9**t); vhat = v/(1-.999**t)
    params = params - lr*mhat/(jnp.sqrt(vhat)+1e-8)
    return params, m, v

# 起始处新旧策略相同：概率比值应全为 1
check = batch
ratio0 = np.exp(np.asarray(normal_logp(actor, check['s'], check['a'])) - check['old_logp'])
assert np.max(np.abs(ratio0 - 1)) < 1e-5
print('PPO ratio invariant passed')

## 5.5 训练并可视化控制效果

为了让 notebook 在 CPU 上也能运行，只使用 3 个策略参数和 3 个价值参数。真实 PPO 会在此处用 MLP（例如 Flax）替换两个解析函数。

In [ ]:
train_rng = np.random.default_rng(1234)
actor = jnp.array([0., 0., -0.5]); critic = jnp.zeros(3)
ma = jnp.zeros_like(actor); va = jnp.zeros_like(actor)
mc = jnp.zeros_like(critic); vc = jnp.zeros_like(critic)
actor_t = critic_t = 0
return_curve, slope_curve = [], []
for update in range(45):
    batch, mean_return = collect_episodes(actor, critic, train_rng, episodes=14)
    return_curve.append(mean_return)
    slope_curve.append(float(actor[0]))
    adv = batch['adv']
    batch['adv'] = (adv-adv.mean())/(adv.std()+1e-8)
    N = len(adv)
    for epoch in range(4):
        indices = train_rng.permutation(N)
        for ids in np.array_split(indices, 4):
            s = jnp.asarray(batch['s'][ids]); a = jnp.asarray(batch['a'][ids])
            lp = jnp.asarray(batch['old_logp'][ids]); A = jnp.asarray(batch['adv'][ids])
            tgt = jnp.asarray(batch['target'][ids])
            _, ga = actor_grad(actor, s, a, lp, A)
            _, gc = critic_grad(critic, s, tgt)
            actor_t += 1; critic_t += 1
            actor, ma, va = adam_step(actor, ga, ma, va, actor_t, lr=.012)
            critic, mc, vc = adam_step(critic, gc, mc, vc, critic_t, lr=.02)
print('learned actor (w,b,log_sigma):', np.round(np.asarray(actor),3))
print('initial / final sampled returns:',round(return_curve[0],2),round(np.mean(return_curve[-5:]),2))
plt.figure(figsize=(7,3))
plt.plot(return_curve, label='sampled episode return')
plt.xlabel('PPO update'); plt.ylabel('Mean return (higher is better)')
plt.legend(); plt.show()
plt.figure(figsize=(7,3))
plt.plot(slope_curve); plt.xlabel('PPO update'); plt.ylabel('policy mean slope w'); plt.show()

## 5.6 固定测试种子，公平比较随机策略与 PPO

训练回报同时包含 exploration 噪声，因此应该单独评估 deterministic mean policy 的物理收益。两种策略测试使用相同的初始状态与过程噪声。

In [ ]:
def test_policy(theta, seed, controlled=True, n=200):
    env_rng = np.random.default_rng(seed)
    theta = np.asarray(theta)
    returns = []
    for i in range(n):
        x = float(np.clip(env_rng.normal(0,1.1),-2,2))
        total = 0.
        for t in range(HORIZON):
            action = float(theta[0]*x + theta[1]) if controlled else 0.
            x, r = env_step(x, action, env_rng)
            total += r
        returns.append(total)
    return float(np.mean(returns))

zero_policy = test_policy(actor, 999, controlled=False)
trained_policy = test_policy(actor, 999, controlled=True)
print(f'No control: {zero_policy:.3f}, trained PPO: {trained_policy:.3f}')
print('Improvement (positive is better):', round(trained_policy-zero_policy,3))

## 5.7 迁移到二维圆柱绕流时需要替换什么？

| 本实验 | 你的毕设 |
|---|---|
| $x_t$（一维位置） | 压力传感器、流场 POD 系数或 latent state |
| $a_t$（一维作用） | 吹吸/旋转等连续控制量 |
| `env_step` | DNS/CFD 或训练好的神经网络代理环境 |
| $-x^2$ | 以减阻为主、兼顾升力波动与作动能耗的奖励 |
| 线性高斯策略 | MLP/CNN 策略，或带历史信息的策略 |
| 28 步轨迹 | 物理时间尺度匹配的控制时域 |

如果你的 surrogate 训练分布之外存在外推误差，PPO 可能会利用代理漏洞获得很高的“预测奖励”，但真实 CFD 并没有减阻。必须采用独立 CFD 闭环验证与交替校正机制。

### 练习（建议按顺序做）

1. 把 `EPS_CLIP` 设为 0.05、0.4，比较策略参数与 return 曲线。
2. 分别设置 `ENTROPY_COEF=0` 和 `0.03`，理解探索与收敛。
3. 把 `policy_mean(theta, x)` 换成双层 MLP，并保持其余训练循环不变。
4. 使用 `jax.vmap` 批量化环境，进一步尝试 `jax.lax.scan` 把 rollout 放入 JIT。
5. 为 PPO ratio=1、GAE 断点以及 action-log-probability 一致性写单元测试。

**延伸阅读**：[CleanRL PPO 实现与关键细节](https://docs.cleanrl.dev/rl-algorithms/ppo/)、[HydroGym JAX 快速开始](https://dynamicslab.github.io/hydrogym/docs/examples/jax/getting_started/)。